<a href="https://colab.research.google.com/github/sadwi18/24P31A4236_DriveReady_OPC_2026/blob/main/Day_5/Day_5_task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q chromadb
import chromadb

FAQS = {
    "shipping":     "We ship pan-India within 3 to 5 business days. Shipping is Rs 60, free on orders above Rs 999.",
    "payment":      "We accept UPI, cards, and Cash on Delivery (COD). COD applies to orders up to Rs 5,000.",
    "returns":      "Returns are accepted within 7 days of delivery for unused items in original packaging.",
    "refunds":      "Refunds are processed to the original payment method within 5 to 7 business days after inspection.",
    "cancellation": "Orders can be cancelled free of charge before dispatch. Once shipped, standard return rules apply.",
    "replacement":  "Damaged or wrong items qualify for a free replacement within 7 days of delivery with unboxing proof.",
    "warranty":     "Select electronics carry a 6-month seller warranty against manufacturing defects.",
    "storage":      "Keep fragile and skincare items in a cool, dry place away from direct sunlight.",
    "gift_wrap":    "Gift wrapping is free on request; custom gift notes are free up to 50 characters.",
    "bulk_orders":  "Bulk orders of 10 or more units need special pricing approval from the support team.",
    "tracking":     "Tracking links are shared via email and SMS once the order is dispatched.",
    "support_hrs":  "Customer support is available Monday to Saturday 9 am to 8 pm, and Sunday 10 am to 6 pm.",
}

client = chromadb.Client()
faq = client.get_or_create_collection(name="shopsphere_faq")
faq.upsert(ids=list(FAQS.keys()), documents=list(FAQS.values()))
print(faq.count(), "facts stored")
print(faq.query(query_texts=["How do I return a damaged product?"], n_results=2)["documents"][0])

12 facts stored
['Damaged or wrong items qualify for a free replacement within 7 days of delivery with unboxing proof.', 'Returns are accepted within 7 days of delivery for unused items in original packaging.']


In [ ]:
from langchain_core.tools import tool

@tool
def search_faq(question: str) -> str:
    """Search ShopSphere's store policies and FAQs: shipping, payment, returns,
    refunds, cancellation, replacement, warranty, gift wrap, and support hours.
    Use for policy or FAQ questions, not for live order status, stock, or catalog prices."""
    results = faq.query(query_texts=[question], n_results=3)
    return "\n".join(results["documents"][0])

In [5]:
import os
from google.colab import userdata
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

@tool
def get_product_info(product: str) -> str:
    """Get price and stock availability for a ShopSphere product."""
    return f"{product}: In stock, Rs 799."

@tool
def track_order(order_id: str) -> str:
    """Track shipment status for a ShopSphere order ID."""
    return f"Order {order_id}: Dispatched, arriving in 2 days."

@tool
def log_return(customer_name: str, order_id: str, item: str, reason: str) -> str:
    """Log a return or support ticket for an order."""
    return f"Return logged for {customer_name} (Order {order_id}, {item}): {reason}."

SYSTEM = """ROLE:   You are the helpful customer support assistant for ShopSphere.
RULES:  Use get_product_info for stock/prices, track_order for order status, and log_return for return/support tickets; never guess them.
        For any policy question, use search_faq and answer ONLY from what it returns.
        If the answer isn't there, say "I'll check with the support team."
        Before log_return, make sure you have customer name, order ID, item, and reason; ask if missing.
FORMAT: Under 60 words, clear and friendly, English or Telugu matching the customer."""

agent = create_agent(
    model=llm,
    tools=[get_product_info, track_order, log_return, search_faq],
    system_prompt=SYSTEM,
    checkpointer=InMemorySaver(),
)

def chat(thread_id: str, text: str) -> str:
    config = {"configurable": {"thread_id": thread_id}}
    try:
        result = agent.invoke({"messages": [{"role": "user", "content": text}]}, config)
        return result["messages"][-1].content
    except Exception as e:
        print("Agent error:", e)
        return "Sorry, I'm a little busy. Please try again in a minute."

print(chat("ravi", "Hi, I'm Ravi. Is shipping free? My cart is Rs 1,200."))
print(chat("ravi", "How many days do I have to return an item?"))
print(chat("sravani", "Can I cancel my order before it ships?"))
print(chat("ravi", "What's my name, and will I pay shipping on my cart?"))

Hi Ravi! Yes, shipping is free for orders above ₹999. Since your cart totals ₹1,200, you’ll enjoy free delivery. Let me know if you need anything else!
You can return an item within 7 days of delivery, as long as it’s unused and in its original packaging. Let me know if you’d like to start a return!
Sure! You can cancel your order for free as long as it hasn’t shipped yet. Once it’s dispatched, the standard return policy applies. If you need help canceling, just let me know your order ID and I’ll take care of it.
Your name is Ravi, and you won’t pay any shipping on your ₹1,200 cart—shipping is free for orders above ₹999.


In [6]:
%%writefile -a tests.csv
Is shipping free above Rs 999?,999
How many days for returns?,7 days
How long do refunds take?,5 to 7 business days
Do you support COD?,COD
Do you sell groceries?,check with the support team

Appending to tests.csv
